# Gemma AST Agent: second benchmark library

Runs the same experiment as the boltons runs (notebook Versions 3 and 4) on a second, different Python library, to test whether the findings generalize.

**Settings:** Accelerator = GPU T4 x2, Internet = On. **Input:** Gemma 4 model, framework Transformers, `gemma-4-e4b-it`.

The code is cloned from GitHub at a pinned commit. The first library in `CANDIDATES` whose own test suite passes cleanly is used.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Code and model

In [ ]:
import glob, os, sys
REPO_URL = 'https://github.com/anisha274/gemma-ast-agent.git'
REPO_COMMIT = '35997dc0f7ebc09945d0004a4bacde0dc1d52f43'
WORK = '/kaggle/working/gemma-ast-agent'
!rm -rf $WORK && git clone -q $REPO_URL $WORK && git -C $WORK checkout -q $REPO_COMMIT
os.chdir(WORK); sys.path.insert(0, WORK)
!git log -1 --format='code commit: %H'
cfgs = [p for p in glob.glob('/kaggle/input/**/config.json', recursive=True) if 'gemma-ast-agent' not in p]
assert cfgs, 'Add a Gemma 4 Transformers model as an input'
MODEL_PATH = os.path.dirname(cfgs[0]); print('model:', MODEL_PATH)
RESULTS = '/kaggle/working/results'

## 2. Install and test the agent (should say 16 passed)

In [ ]:
!pip install -q -U transformers accelerate pytest hypothesis freezegun pytz
!python -m pytest -q tests

## 3. Choose the second library and plant 50 bugs (CPU work)

In [ ]:
import subprocess
from ast_agent.runner import run_tests
CANDIDATES = [
    ('https://github.com/pallets/click.git', '8.5.0'),
    ('https://github.com/dateutil/dateutil.git', '3.0.0'),
    ('https://github.com/more-itertools/more-itertools.git', 'v11.2.1'),
]
TEST_CMD = 'python -m pytest -q --tb=short -p no:cacheprovider'
LIB = None
for url, tag in CANDIDATES:
    name = url.rstrip('/').split('/')[-1].removesuffix('.git')
    path = f'benchmarks/repos/{name}'
    subprocess.run(['rm', '-rf', path])
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', tag, url, path], check=True)
    clean = run_tests(TEST_CMD, path, timeout=600)
    print(f'{name} {tag}: clean test suite', 'PASSES' if clean.passed else 'FAILS', f'({clean.duration_s:.0f}s)')
    if clean.passed:
        LIB, LIB_TAG = name, tag
        break
    print(clean.output[-800:])
assert LIB, 'No candidate library passed its own tests'
print('Using', LIB, LIB_TAG)

In [ ]:
!python -m ast_agent.mutate --repo benchmarks/repos/{LIB} --test-cmd "{TEST_CMD}" --out benchmarks/{LIB} --max-tasks 50 --seed 0 --prefix {LIB}
TASKS = f'benchmarks/{LIB}/tasks.jsonl'

Harness check: applying each known fix must resolve every task.

In [ ]:
!python -m ast_agent.eval --tasks $TASKS --backend gold --conditions full --out $RESULTS/{LIB}_gold 2>&1 | tail -8

## 4. Experiment with Gemma 4 E4B

In [ ]:
!python -m ast_agent.eval --tasks $TASKS --backend hf --model "$MODEL_PATH" \
  --dtype float16 --conditions file,ast_graph,full,oracle --max-new-tokens 1024 \
  --out $RESULTS/{LIB}_e4b 2>&1 | grep -v ' INFO '

In [ ]:
from IPython.display import Markdown
print(LIB, LIB_TAG)
Markdown(open(f'{RESULTS}/{LIB}_e4b/summary.md').read())